# 《로봇 없이 시작하는 Physical AI 따라하기》 — Colab 학습 노트북

GPU가 없는 독자를 위한 노트북입니다. 책의 **6-5(데이터셋 만들기) → 6-7(ACT 학습) → 6-8(평가)** 을 Colab에서 그대로 실행합니다.
7부의 SmolVLA 파인튜닝 명령도 마지막에 있습니다.

- 책: https://wikidocs.net/book/21363
- 예제 코드: https://github.com/ady95/physicalai_tutorial

**먼저 할 일**: 메뉴에서 런타임 → 런타임 유형 변경 → 하드웨어 가속기를 **T4 GPU**로 바꾸세요.

> 이 노트북의 명령들은 Ubuntu 22.04 + NVIDIA GPU 환경에서 검증했습니다. Colab 환경 자체에서는 필자가 실행해 보지 않았으므로,
> 셀이 실패하면 책의 부록 E와 저장소 이슈를 참고하세요.


## 1. GPU 확인


In [ ]:
!nvidia-smi


## 2. 예제 저장소와 환경 설치

uv가 저장소의 uv.lock에 적힌 정확한 버전으로 설치합니다. Colab의 Python 버전과 무관하게 Python 3.12를 알아서 내려받습니다.
PyTorch와 LeRobot을 받으므로 5분 남짓 걸립니다.


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
!uv --version


In [ ]:
%cd /content
!rm -rf physicalai_tutorial
!git clone -q https://github.com/ady95/physicalai_tutorial.git
%cd /content/physicalai_tutorial
!uv sync


설치를 확인합니다. 책 2-2의 출력과 같아야 합니다.


In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'        # 화면 없이 렌더링 (책 2-3)
os.environ['SVT_LOG'] = '0'            # 영상 인코더 로그 끄기
!.venv/bin/python ch02/check_env.py


## 3. 시연 데이터셋 만들기 (책 6-5)

3부의 Rule 기반 Pick and Place 프로그램이 시연자입니다. 블록 위치를 무작위로 바꿔 가며 기록합니다.

이 셀은 CPU만 쓰고 한 번에 한 Episode씩 진행합니다. 필자의 서버(36코어)에서 150회에 16분이 걸렸고,
코어가 적은 Colab에서는 30분 이상 걸릴 수 있습니다. 세션 시간이 걱정되면 아래 두 가지 중 하나를 고르세요.

- **그대로 진행**: `EPISODES = 150`. 책과 같은 조건이며 성공률 65%를 기대할 수 있습니다.
- **학습을 건너뛰기**: [릴리스의 체크포인트](https://github.com/ady95/physicalai_tutorial/releases/tag/v0.1-checkpoints)를 받아 5번 평가부터 진행합니다.

Episode 수를 줄이는 것은 권하지 않습니다. 책의 실측에서 43 Episode는 성공률이 10%에 그쳤습니다.


In [ ]:
EPISODES = 150   # 저장되는 Episode 는 이보다 적습니다 (실패한 시연은 버립니다)
# --overwrite: 이 셀을 다시 실행하면 이전 데이터셋을 지우고 새로 기록합니다
!.venv/bin/python ch06/record_demos.py --episodes {EPISODES} --seed 1 --overwrite \
    --root outputs/datasets/sim --repo-id colab/so101_pickplace_sim


데이터셋 구조를 확인합니다 (책 6-4).


In [ ]:
!.venv/bin/python ch06/inspect_dataset.py --root outputs/datasets/sim --repo-id colab/so101_pickplace_sim


## 4. ACT 학습 (책 6-7)

RTX 3060에서 10,000스텝에 약 30분이었습니다. T4는 그보다 느립니다.
Colab 세션이 끊겨도 `--save_freq` 로 남긴 체크포인트에서 이어서 학습할 수 있습니다. 마지막 체크포인트의 설정 파일을 함께 지정해야 합니다.

```
!.venv/bin/lerobot-train \
    --config_path=outputs/train/act/checkpoints/last/pretrained_model/train_config.json \
    --resume=true
```

세션이 새로 시작되면 outputs 폴더도 사라지므로, 이어서 학습하려면 체크포인트 폴더를 Google Drive 에 복사해 두세요.


In [ ]:
STEPS = 10000
!.venv/bin/lerobot-train \
    --dataset.repo_id=colab/so101_pickplace_sim --dataset.root=outputs/datasets/sim \
    --policy.type=act --policy.chunk_size=50 --policy.n_action_steps=50 \
    --policy.device=cuda --policy.push_to_hub=false \
    --output_dir=outputs/train/act --job_name=act \
    --steps={STEPS} --batch_size=16 --save_freq=2500 --log_freq=500 --wandb.enable=false


## 5. 평가 (책 6-8)

학습한 Policy 로 20 Episode 를 실행하고 성공률을 잽니다. 책의 실측은 117 Episode 데이터셋, 10,000스텝에서 65% 였습니다.


In [ ]:
!.venv/bin/python ch06/eval_act.py \
    --checkpoint outputs/train/act/checkpoints/last/pretrained_model \
    --episodes 20 --video --out outputs/colab_act_eval.mp4


평가 영상을 노트북에서 재생합니다.


In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open('outputs/colab_act_eval.mp4', 'rb').read()).decode()
HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')


## 6. 체크포인트 내려받기

세션이 끝나면 /content 의 파일은 사라집니다. 필요한 것을 내려받거나 Google Drive 에 복사하세요.

두 가지를 구분합니다. **평가용**은 가중치와 정규화 통계가 든 `pretrained_model` 만 있으면 되고(책 6-8, 7-4), **학습 재개용**은 optimizer 와 스케줄러 상태가 든 `training_state` 까지 필요합니다(책 6-7, 부록 D).

In [ ]:
# 평가용 (작음): pretrained_model 만
!tar czf act_checkpoint.tar.gz -C outputs/train/act/checkpoints/last pretrained_model
from google.colab import files
files.download('act_checkpoint.tar.gz')

Google Drive 에 저장하려면 아래를 사용합니다.

학습 재개용으로는 두 가지를 함께 보존해야 합니다. **체크포인트 폴더 전체**와 **데이터셋 폴더**입니다. 체크포인트의 `train_config.json` 에 데이터셋 경로가 기록되어 있어, 재개할 때 그 경로에 데이터셋이 없으면 `FileNotFoundError: outputs/datasets/sim/meta/info.json` 로 멈춥니다. 150 episode 데이터셋이 약 80 MB 라 Drive 에 두는 편이 다시 기록하는 것(약 16분)보다 빠르고, 같은 데이터라는 것도 보장됩니다. 체크포인트 폴더는 optimizer 상태까지 담겨 ACT 기준 약 0.6 GB 입니다.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # 평가용
# !cp act_checkpoint.tar.gz /content/drive/MyDrive/
#
# # 학습 재개용: 체크포인트 폴더 전체 + 데이터셋
# !mkdir -p /content/drive/MyDrive/physicalai_act
# !cp -r outputs/train/act/checkpoints /content/drive/MyDrive/physicalai_act/
# !cp -r outputs/datasets/sim /content/drive/MyDrive/physicalai_act/
#
# # 새 세션에서 이어서 학습하기: 둘 다 같은 경로로 되돌린 뒤 resume
# !mkdir -p outputs/train/act outputs/datasets
# !cp -r /content/drive/MyDrive/physicalai_act/checkpoints outputs/train/act/
# !cp -r /content/drive/MyDrive/physicalai_act/sim outputs/datasets/
# !.venv/bin/lerobot-train \
#     --config_path=outputs/train/act/checkpoints/last/pretrained_model/train_config.json \
#     --resume=true

## 7. (선택) SmolVLA 파인튜닝 (책 7-4)

같은 데이터셋으로 VLA 를 파인튜닝합니다. 모델이 4.5억 파라미터라 ACT 보다 오래 걸립니다(RTX 3060 기준 10,000스텝 약 60분).
카메라 이름을 사전학습 모델이 기대하는 이름으로 바꿔 주는 `--rename_map` 이 필요합니다(책 7-4, 부록 E).


In [ ]:
RENAME = '{"observation.images.top": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}'
!.venv/bin/lerobot-train \
    --policy.path=lerobot/smolvla_base \
    --dataset.repo_id=colab/so101_pickplace_sim --dataset.root=outputs/datasets/sim \
    --rename_map='{RENAME}' \
    --policy.device=cuda --policy.push_to_hub=false \
    --output_dir=outputs/train/smolvla --job_name=smolvla \
    --steps=10000 --batch_size=8 --save_freq=2500 --log_freq=250 --wandb.enable=false


In [ ]:
!.venv/bin/python ch07/eval_smolvla.py \
    --checkpoint outputs/train/smolvla/checkpoints/last/pretrained_model \
    --episodes 20 --video --out outputs/colab_smolvla_eval.mp4


---

학습을 건너뛰고 평가만 해 보고 싶다면 [릴리스 페이지](https://github.com/ady95/physicalai_tutorial/releases/tag/v0.1-checkpoints)의 체크포인트를 내려받아 4~5번 대신 사용하세요.
